# Theorem 16 — source support is not conditional identifiability

**Formal source:** [`../16_source_supported_missing_identifiability.md`](../16_source_supported_missing_identifiability.md)

This Notebook is an executable finite witness, not the general proof. Passing it supports implementation consistency only; it does not establish learned-model or real-PHM evidence.

In [ ]:
import math
import itertools
import numpy as np
np.set_printoptions(precision=6, suppress=True)


def four_way(projectors, domain_index, atol=1e-9):
    ps = [np.asarray(p, float) for p in projectors]
    dimension = ps[0].shape[0]
    summed = sum(ps)
    values, vectors = np.linalg.eigh((summed + summed.T) / 2)
    basis = vectors[:, np.isclose(values, len(ps), atol=atol)]
    shared = basis @ basis.T if basis.size else np.zeros((dimension, dimension))
    basis = vectors[:, values > atol]
    union = basis @ basis.T if basis.size else np.zeros((dimension, dimension))
    observed = ps[domain_index]
    blocks = [shared, observed - shared, union - observed, np.eye(dimension) - union]
    for projector in blocks:
        np.testing.assert_allclose(projector, projector.T, atol=1e-8)
        np.testing.assert_allclose(projector @ projector, projector, atol=1e-8)
    for index, left in enumerate(blocks):
        for right in blocks[index + 1:]:
            np.testing.assert_allclose(left @ right, 0, atol=1e-8)
    np.testing.assert_allclose(sum(blocks), np.eye(dimension), atol=1e-8)
    return blocks


def normal_pdf(x, mean, standard_deviation):
    return np.exp(-0.5 * ((x - mean) / standard_deviation) ** 2) / (
        math.sqrt(2 * math.pi) * standard_deviation
    )

In [ ]:
positive_world = np.array([[0, 0], [1, 1]])
negative_world = np.array([[0, 1], [1, 0]])
for column in [0, 1]:
    np.testing.assert_array_equal(np.sort(positive_world[:, column]), np.sort(negative_world[:, column]))

def conditional(table):
    return {int(c): float(table[table[:, 0] == c, 1].mean()) for c in [0, 1]}

positive_conditional = conditional(positive_world)
negative_conditional = conditional(negative_world)
assert positive_conditional == {0: 0.0, 1: 1.0}
assert negative_conditional == {0: 1.0, 1: 0.0}
print({"unpaired_marginals_equal": True, "paired_positive": positive_conditional, "paired_negative": negative_conditional})

In [ ]:
print('THEORY_DEMO_PASS::16_source_supported_missing_identifiability')
print('evidence_level: constructive_or_numerical_witness')
print('formal_claim_supported: false')